In [1]:
import requests

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "nuccore",
    "id": "NM_000793.6",
    "rettype": "fasta",
    "retmode": "text"
}

response = requests.get(url, params=params)
print(response.text)
with open("DIO2.fasta", "w") as fh:
    fh.write(response.text)

>NM_000793.6 Homo sapiens iodothyronine deiodinase 2 (DIO2), transcript variant 2, mRNA
GTGAGACTGGTCTAGCCTCCCAGCCTACATCTTTCTTCAGTGCTGGATGCTTCCTGCCCTCAAACATCGG
ACTCCAAGTTCTTCAGTTTTGAGACTCAGACTGACTGTCCTTGCTCCTCAAGCTTGCAGACAGCCTATTG
TGGGACGTTGTGATCATTAGGTAGCCTGGAAAAAAAAGAGAAGCAACAGTTCCTGAGGGAAGAATGCATT
AACAACTCGGATGGGTGCTGTCTGCCAGCACTGCAGAGGAACCTACGGAGAGGGCTGCTGGGAGAGAATG
CTCATACCTCCCAAGGCGTCAGCCTGCCCTCCCGCCCCCAAGTTGCTTTGCTCAAGAGGGTGAAGGGGAA
CCAGAGCGCACAAGGGAACTGACTCAGGAGGCAGAGAAGATGGGCATCCTCAGCGTAGACTTGCTGATCA
CACTGCAAATTCTGCCAGTTTTTTTCTCCAACTGCCTCTTCCTGGCTCTCTATGACTCGGTCATTCTGCT
CAAGCACGTGGTGCTGCTGTTGAGCCGCTCCAAGTCCACTCGCGGAGAGTGGCGGCGCATGCTGACCTCA
GAGGGACTGCGCTGCGTCTGGAAGAGCTTCCTCCTCGATGCCTACAAACAGGTGAAATTGGGTGAGGATG
CCCCCAATTCCAGTGTGGTGCATGTCTCCAGTACAGAAGGAGGTGACAACAGTGGCAATGGTACCCAGGA
GAAGATAGCTGAGGGAGCCACATGCCACCTTCTTGACTTTGCCAGCCCTGAGCGCCCACTAGTGGTCAAC
TTTGGCTCAGCCACTTGACCTCCTTTCACGAGCCAGCTGCCAGCCTTCCGCAAACTGGTGGAAGAGTTCT
CCTCAGTGGCTGACTTCCTGCTGGTCTACATTGATGAGGCTCATCCATCAGATGGCTGGG

In [2]:
from Bio import SeqIO
from io import StringIO
import requests

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "nuccore",
    "id": "NM_000793.6",
    "rettype": "gb",
    "retmode": "text"
}

response = requests.get(url, params=params)

with open("DIO2.gb", "w") as fh:
    fh.write(response.text)
record = SeqIO.read(StringIO(response.text), "genbank")
 
print("ACCESSION:", record.id)
print("ORGANISM:", record.annotations.get("organism", "N/A"))
print("LENGTH:", len(record.seq), "bp")
 
for feature in record.features:
    if feature.type == "CDS":
        print("CDS:", feature.location)
        print("PROTEIN ID:", feature.qualifiers.get("protein_id", ["N/A"])[0])
        break  

ACCESSION: NM_000793.6
ORGANISM: Homo sapiens
LENGTH: 6374 bp
CDS: [389:1187](+)
PROTEIN ID: NP_000784.3


In [3]:
import requests

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

proteins = {
    "DIO2": "NP_000784.3"
}

for gene, protein_id in proteins.items():
    params = {
        "db": "protein",
        "id": protein_id,
        "rettype": "fasta",
        "retmode": "text"
    }

    response = requests.get(url, params=params)

    filename = f"{gene}_protein.fasta"

    with open(filename, "w") as fh:
        fh.write(response.text)

    print(f"{gene}: {protein_id}")
    print(response.text.splitlines()[0])

DIO2: NP_000784.3
>NP_000784.3 type II iodothyronine deiodinase isoform b [Homo sapiens]


In [1]:
import requests

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

proteins = {
    "DIO2": "NP_000784.3"
}

for gene, protein_id in proteins.items():

    params = {
        "db": "protein",
        "id": protein_id,
        "rettype": "fasta",
        "retmode": "text"
    }

    response = requests.get(url, params=params)

    filename = f"{gene}_protein.fasta"

    with open(filename, "w") as fh:
        fh.write(response.text)

    #reads a fasta file and ignores headers and removes spaces
    sequence = "".join(
        line.strip()
        for line in response.text.splitlines()
        if not line.startswith(">")
    )

    print(gene)
    print("Protein ID:", protein_id)
    print("Length:", len(sequence), "residues")
    print()

DIO2
Protein ID: NP_000784.3
Length: 265 residues



In [5]:
with open("DIO2_sequence.txt") as fh:
    sequence = fh.read().strip()

cds = sequence[389:1187]

print("CDS length:", len(cds))
print("Divisible by 3:", len(cds) % 3 == 0)

CDS length: 798
Divisible by 3: True


In [7]:
from Bio import SeqIO

record = SeqIO.read("DIO2.fasta", "fasta")
sequence = str(record.seq).upper()

cds = sequence[389:1187]

#defining the directory
codon_table = {
    "TTT":"F", "TTC":"F", "TTA":"L", "TTG":"L",
    "TCT":"S", "TCC":"S", "TCA":"S", "TCG":"S",
    "TAT":"Y", "TAC":"Y", "TAA":"*", "TAG":"*",
    "TGT":"C", "TGC":"C", "TGA":"*", "TGG":"W",

    "CTT":"L", "CTC":"L", "CTA":"L", "CTG":"L",
    "CCT":"P", "CCC":"P", "CCA":"P", "CCG":"P",
    "CAT":"H", "CAC":"H", "CAA":"Q", "CAG":"Q",
    "CGT":"R", "CGC":"R", "CGA":"R", "CGG":"R",

    "ATT":"I", "ATC":"I", "ATA":"I", "ATG":"M",
    "ACT":"T", "ACC":"T", "ACA":"T", "ACG":"T",
    "AAT":"N", "AAC":"N", "AAA":"K", "AAG":"K",
    "AGT":"S", "AGC":"S", "AGA":"R", "AGG":"R",

    "GTT":"V", "GTC":"V", "GTA":"V", "GTG":"V",
    "GCT":"A", "GCC":"A", "GCA":"A", "GCG":"A",
    "GAT":"D", "GAC":"D", "GAA":"E", "GAG":"E",
    "GGT":"G", "GGC":"G", "GGA":"G", "GGG":"G"
}

dio2_protein = ""

#Takes 3 codons and then translates the codon to the amino acid from the codon table
for i in range(0, len(cds), 3):
    codon = cds[i:i+3]
    amino_acid = codon_table[codon]
    dio2_protein += amino_acid

print("First 30 residues:", dio2_protein[:30])
print("Total translated length:", len(dio2_protein), "residues")
print("Translated protein:", dio2_protein)    


First 30 residues: MGILSVDLLITLQILPVFFSNCLFLALYDS
Total translated length: 266 residues
Translated protein: MGILSVDLLITLQILPVFFSNCLFLALYDSVILLKHVVLLLSRSKSTRGEWRRMLTSEGLRCVWKSFLLDAYKQVKLGEDAPNSSVVHVSSTEGGDNSGNGTQEKIAEGATCHLLDFASPERPLVVNFGSAT*PPFTSQLPAFRKLVEEFSSVADFLLVYIDEAHPSDGWAIPGDSSLSFEVKKHQNQEDRCAAAQQLLERFSLPPQCRVVADRMDNNANIAYGVAFERVCIVQRQKIAYLGGKGPFSYNLQEVRHWLEKNFSKR*


In [8]:
# Reading deposited DIO2 protein
with open("DIO2_protein.fasta") as fh:
    dio2_deposited = "".join(
        line.strip()
        for line in fh
        if not line.startswith(">")
    )

# Comparing the two sequences
dio2_identical = dio2_protein == dio2_deposited

if dio2_identical:
    print("The two protein sequences are IDENTICAL.")
    dio2_first_mismatch = "None"

else:
    print("The two protein sequences are NOT identical.")

    dio2_first_mismatch = "None"

    for i in range(min(len(dio2_protein), len(dio2_deposited))):
        if dio2_protein[i] != dio2_deposited[i]:
            dio2_first_mismatch = i + 1
            print("First difference at position:", dio2_first_mismatch)
            print("Your translation:", dio2_protein[i])
            print("Deposited protein:", dio2_deposited[i])
            break

The two protein sequences are NOT identical.
First difference at position: 133
Your translation: *
Deposited protein: U


In [3]:
#from Bio import SeqIO
#extracting the file and defining it as a fasta file
#record = SeqIO.read("DIO2.fasta", "fasta")
#gets the sequence from biopython record and convert it to uppercase
#dio2_sequence = str(record.seq).upper()

cds = dio2_sequence[389:1187]

print("DIO2 CDS:", len(cds), "nt")

DIO2 CDS: 798 nt


In [4]:
from Bio.Seq import Seq

# Translate the same DIO2 CDS using Biopython
biopython_dio2 = str(Seq(cds).translate())

# Compare lengths
print("My translation length:", len(dio2_protein))
print("Biopython length:", len(biopython_dio2))

# Compare the two proteins
if dio2_protein == biopython_dio2:
    print("My translation and Biopython AGREE.")

else:
    print("My translation and Biopython DO NOT AGREE.")

    for i in range(min(len(dio2_protein), len(biopython_dio2))):
        if dio2_protein[i] != biopython_dio2[i]:
            print("First difference at position:", i + 1)
            print("My translation:", dio2_protein[i])
            print("Biopython:", biopython_dio2[i])
            break

NameError: name 'dio2_protein' is not defined

In [10]:
translated_length = len(dio2_protein)
deposited_length = len(dio2_deposited)
difference = abs(translated_length - deposited_length)

print("Translated length:", translated_length, "residues")
print("Deposited length:", deposited_length, "residues")
print("Difference:", difference, "residues")

Translated length: 266 residues
Deposited length: 265 residues
Difference: 1 residues


In [15]:
with open("DIO2_results.txt", "w") as fh:
    fh.write("DIO2\n")
    fh.write("Category A -selenoprotein\n")
    fh.write(str(len(cds)) + "\n")
    fh.write(str(len(dio2_protein)) + "\n")
    fh.write(str(len(dio2_deposited)) + "\n")
    fh.write(("Y" if dio2_identical else "N") + "\n")
    fh.write(str(dio2_first_mismatch) + "\n")

print("DIO2_results.txt created")

DIO2_results.txt created


In [2]:
def show_summary(files):

    print(f"{'Gene':<12}{'Category':<30}{'CDS Length':<15}"
          f"{'Translated':<15}{'Deposited':<15}"
          f"{'Identical':<12}{'First Mismatch'}")

    for filename in files:

        with open(filename) as f:
            data = [line.strip() for line in f]

        print(f"{data[0]:<12}{data[1]:<30}{data[2]:<15}"
              f"{data[3]:<15}{data[4]:<15}{data[5]:<12}{data[6]}")
show_summary([
    "DIO2_results.txt"
])

Gene        Category                      CDS Length     Translated     Deposited      Identical   First Mismatch
DIO2        Category A -selenoprotein     798            266            265            N           133
